https://alasdair.github.io/

So it seems like --smt output combined with some grep trickery could work
@property


In [ ]:
%%file /tmp/test_sail.sail

default Order dec
$include <prelude.sail>

val EXTZ : forall 'n 'm, 'm >= 'n. (implicit('m), bits('n)) -> bits('m)
function EXTZ(m, v) = sail_zero_extend(v, m)

val EXTS : forall 'n 'm, 'm >= 'n. (implicit('m), bits('n)) -> bits('m)
function EXTS(m, v) = sail_sign_extend(v, m)

type xlen       : Int = 64
type xlen_bytes : Int = 8
type xlenbits         = bits(xlen)

type regbits = bits(5)

register PC : xlenbits
register nextPC : xlenbits

register Xs : vector(32, dec, xlenbits)

val rX : regbits -> xlenbits
function rX(r) =
  match r {
    0b00000 => EXTZ(0x0),
    _ => Xs[unsigned(r)]
  }

val wX : (regbits, xlenbits) -> unit
function wX(r, v) =
  if r != 0b00000 then {
     Xs[unsigned(r)] = v;
  }

overload X = {rX, wX}


Writing /tmp/test_sail.sail


In [26]:
%%file /tmp/test_sail.sail

default Order dec
$include <prelude.sail>

val foo : bits(32) -> bits(32)
function foo(x) = x + 1

$property
function foo_rel(x : bits(32), y : bits(32)) -> bool = foo(x) == y

Overwriting /tmp/test_sail.sail


In [29]:
! sail /tmp/test_sail.sail --smt -o myprefix

omd: debug mode activated because DEBUG is set, you can deactivate the mode by unsetting DEBUG or by setting OMD_DEBUG to the string "false".
omd: debug mode activated because DEBUG is set, you can deactivate the mode by unsetting DEBUG or by setting OMD_DEBUG to the string "false".


In [30]:
! cat ./myprefix_foo_rel.smt2

(declare-datatypes ((Unit 0)) (((unit))))
(declare-datatypes ((Bits 0)) (((Bits (len (_ BitVec 8)) (contents (_ BitVec 64))))))
(declare-datatypes ((Zexception 0)) (((z__dummy_exnz3 (unz__dummy_exnz3 Unit)))))
(declare-const zz40/1 (_ BitVec 32))
(declare-const zz41/1 (_ BitVec 32))
(assert (not (= (bvadd zz40/1 #x00000001) zz41/1)))
(check-sat)


In [ ]:
import re
# maycj from file

f = open("./myprefix_foo_rel.smt2")
lines = f.read()


re.findall(r"\(declare-const (\S*) ", lines)

import z3

spec = z3.parse_smt2_file("./myprefix_foo_rel.smt2")
spec[0]
from kdrag.all import *
consts = {c.decl().name() : c for c in kd.utils.consts(spec[0])}
args = [consts[name] for name in re.findall(r"\(declare-const (\S*) ", lines)]
args

smt.Lambda(args, spec[0])



Lambda([zz40/1, zz41/1], Not(zz40/1 + 1 == zz41/1))

In [ ]:
def to_sail_sort(s):
    if isinstance(s, z3.BitVecSortRef):
        return f"bits({s.size()})"
    elif s == z3.BoolSort():
        return "bool"

def run_sail(name, file):
    import subprocess
    stub = f"""
$include {file}
$property
function knuckledummy({', '.join(f'arg{i} : bits(32)' for i in range(10))}) -> bool = {name}({",".join("arg")}) = out



    """
    subprocess.run(["sail", "/tmp/test_sail.sail", "--smt", "-o", "myprefix"], check=True)




In [62]:
%%file /tmp/test_sail.sail

default Order dec
$include <prelude.sail>

val EXTZ : forall 'n 'm, 'm >= 'n. (implicit('m), bits('n)) -> bits('m)
function EXTZ(m, v) = sail_zero_extend(v, m)

val EXTS : forall 'n 'm, 'm >= 'n. (implicit('m), bits('n)) -> bits('m)
function EXTS(m, v) = sail_sign_extend(v, m)

type xlen       : Int = 64
type xlen_bytes : Int = 8
type xlenbits         = bits(xlen)

val MEMr = impure { lem: "MEMr", coq: "MEMr",  smt : "myread", _ : "read_ram",} : forall 'n 'm, 'n >= 0.
   (int('m), int('n), bits('m), bits('m)) -> bits(8 * 'n)

val read_mem : forall 'n, 'n >= 0. (xlenbits, int('n)) -> bits(8 * 'n)
function read_mem(addr, width) =
    MEMr(sizeof(xlen), width, EXTZ(0x0), addr)

$property
function foo_rel(x : bits(64), y : bits(64)) -> bool = read_mem(x, 8) == y

Overwriting /tmp/test_sail.sail


In [63]:
! sail /tmp/test_sail.sail --smt -o myprefix

omd: debug mode activated because DEBUG is set, you can deactivate the mode by unsetting DEBUG or by setting OMD_DEBUG to the string "false".
omd: debug mode activated because DEBUG is set, you can deactivate the mode by unsetting DEBUG or by setting OMD_DEBUG to the string "false".
Error:
Code generated nearby:
/tmp/test_sail.sail:20.4-46:
20 |    MEMr(sizeof(xlen), width, EXTZ(0x0), addr)
   |    ^----------------------------------------^
   | No generator zz7mzJzK64zCz0z7nzJzK8#MEMr


In [13]:
! cd /tmp && sail /tmp/test_sail.sail --ddump-tc-ast

omd: debug mode activated because DEBUG is set, you can deactivate the mode by unsetting DEBUG or by setting OMD_DEBUG to the string "false".
omd: debug mode activated because DEBUG is set, you can deactivate the mode by unsetting DEBUG or by setting OMD_DEBUG to the string "false".
val internal_pick = monadic {_: "internal_pick"}: forall ('a : Type).
  list('a) -> 'a

val undefined_bool = monadic {_: "undefined_bool"}: unit -> bool

val undefined_bit = monadic {_: "undefined_bit"}: unit -> bit

val undefined_int = monadic {_: "undefined_int"}: unit -> int

val undefined_nat = monadic {_: "undefined_nat"}: unit -> nat

val undefined_real = monadic {_: "undefined_real"}: unit -> real

val undefined_string = monadic {_: "undefined_string"}: unit -> string

val undefined_list = monadic {_: "undefined_list"}: forall ('a : Type).
  'a -> list('a)

val undefined_range = monadic {_: "undefined_range"}: forall ('n : Int) ('m : Int).
  (int('n), int('m)) -> range('n, 'm)

val undefined_vector =